# v2 Phase 2: File-Level Feature Engineering
This notebook implements Oh et al.'s 3-step methodology to calculate file-level behavioral features:
1. BDP Detection (Basic_Info_Changed + File_Closed within 1s)
2. Timestamp Verification (compare event time vs SI timestamps)
3. Tunneling Detection (15-second window, different FRNs, same path)
4. Behavior Counter Calculation (0-5 scoring)


**Input Files**
* Filtered UsnJrnl
* Filtered LogFile

**Output Files**
* UsnJrnl file features
* LogFile file features 
* Combine features 

In [1]:
# Cell 1: Import libraries and set paths

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Paths
BASE_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis')
USNJRNL_FILTERED_PATH = BASE_DIR / 'notebooks/LW Testing Pipeline/Outputs/Filtered Version with Old Notebook 2/LoneWolf_UsnJrnl_filtered.csv'
LOGFILE_FILTERED_PATH = BASE_DIR / 'notebooks/LW Testing Pipeline/Outputs/Filtered Version with Old Notebook 2/LoneWolf_LogFile_filtered.csv'

# Output paths
OUTPUT_DIR = BASE_DIR / 'notebooks/LW Testing Pipeline/Outputs/v2 filtered'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USNJRNL_FEATURES_PATH = OUTPUT_DIR / 'LoneWolf_UsnJrnl_file_features.csv'
LOGFILE_FEATURES_PATH = OUTPUT_DIR / 'LoneWolf_LogFile_file_features.csv'
COMBINED_FEATURES_PATH = OUTPUT_DIR / 'LoneWolf_combined_features.csv'

print(f"UsnJrnl input: {USNJRNL_FILTERED_PATH}")
print(f"LogFile input: {LOGFILE_FILTERED_PATH}")
print(f"Output directory: {OUTPUT_DIR}")


UsnJrnl input: /Users/soni/Github/Digital-Detectives_Thesis/notebooks/LW Testing Pipeline/Outputs/Filtered Version with Old Notebook 2/LoneWolf_UsnJrnl_filtered.csv
LogFile input: /Users/soni/Github/Digital-Detectives_Thesis/notebooks/LW Testing Pipeline/Outputs/Filtered Version with Old Notebook 2/LoneWolf_LogFile_filtered.csv
Output directory: /Users/soni/Github/Digital-Detectives_Thesis/notebooks/LW Testing Pipeline/Outputs/v2 filtered


# Part 1: UsnJrnl File-Level Features

In [2]:
# Cell 2: Load filtered UsnJrnl

print("Loading filtered UsnJrnl...")
usnjrnl = pd.read_csv(USNJRNL_FILTERED_PATH, low_memory=False)

print(f"Total events: {len(usnjrnl):,}")
print(f"Unique files (FileReferenceNumber): {usnjrnl['FileReferenceNumber'].nunique():,}")
print(f"\nColumns: {list(usnjrnl.columns)}")


Loading filtered UsnJrnl...
Total events: 16,647
Unique files (FileReferenceNumber): 11,813

Columns: ['TimeStamp(UTC+8)', 'USN', 'File/Directory Name', 'FullPath', 'EventInfo', 'SourceInfo', 'FileAttribute', 'FileReferenceNumber', 'FileName_MFT', 'SI_CreationTime_Formatted', 'SI_ModifiedTime_Formatted', 'SI_AccessedTime_Formatted', 'SI_MFTModifiedTime_Formatted', 'EventTime_Formatted', 'MFT_RecordNumber']


In [3]:
# Cell 3: Timestamp parsing helper function

def normalize_timestamp(ts_str):
    """
    Handle CSV timestamp mangling:
    - Colon separator instead of period for milliseconds
    - Variable length milliseconds (3-4 digits)
    - Returns normalized format: MM/DD/YYYY HH:MM:SS.mmm
    """
    if pd.isna(ts_str):
        return None
    
    parts = str(ts_str).split(' ')
    if len(parts) != 2:
        return None
    
    date_part, time_part = parts[0], parts[1]
    time_components = time_part.split(':')
    
    if len(time_components) < 3:
        return None
    
    # Take first 3 digits of milliseconds
    ms = time_components[3][:3] if len(time_components) == 4 else '000'
    
    return f"{date_part} {time_components[0]}:{time_components[1]}:{time_components[2]}.{ms}"

print("Timestamp normalization function defined")


Timestamp normalization function defined


In [4]:
# Cell 4: Parse all timestamps

print("Parsing timestamps...")

# Parse EventTime (2-digit year from UsnJrnl, needs normalization)
usnjrnl['EventTime'] = pd.to_datetime(
    usnjrnl['TimeStamp(UTC+8)'].apply(normalize_timestamp),
    format='%m/%d/%y %H:%M:%S.%f',
    errors='coerce'
)

# Parse MFT timestamps (4-digit year, ALREADY formatted with period separator)
for col in ['SI_CreationTime', 'SI_ModifiedTime', 'SI_AccessedTime']:
    if f'{col}_Formatted' in usnjrnl.columns:
        # MFT timestamps already have period separator, parse directly
        usnjrnl[col] = pd.to_datetime(
            usnjrnl[f'{col}_Formatted'],
            format='%m/%d/%Y %H:%M:%S.%f',
            errors='coerce'
        )
    elif col in usnjrnl.columns:
        usnjrnl[col] = pd.to_datetime(usnjrnl[col], errors='coerce')

print(f"EventTime parsed: {usnjrnl['EventTime'].notna().sum():,} / {len(usnjrnl):,}")
print(f"SI_CreationTime parsed: {usnjrnl['SI_CreationTime'].notna().sum():,} / {len(usnjrnl):,}")
print(f"\nSample parsed timestamps:")
print(usnjrnl[['TimeStamp(UTC+8)', 'EventTime', 'SI_CreationTime']].head(3))


Parsing timestamps...
EventTime parsed: 16,647 / 16,647
SI_CreationTime parsed: 16,645 / 16,647

Sample parsed timestamps:
         TimeStamp(UTC+8)               EventTime         SI_CreationTime
0  04/01/18 16:57:12:5712 2018-04-01 16:57:12.571 2017-09-29 16:45:11.805
1  04/01/18 16:57:12:5712 2018-04-01 16:57:12.571 2017-09-29 16:45:11.805
2   04/01/18 17:02:12:212 2018-04-01 17:02:12.212 2017-09-29 16:45:11.805


In [5]:
# Cell 5: Step 1 - BDP Detection (Basic_Info_Changed + File_Closed within 1s) - OPTIMIZED

print("\nStep 1: BDP Detection (optimized)...")

# Identify Basic_Info_Changed events
bdp_candidates = usnjrnl[
    usnjrnl['EventInfo'].str.contains('Basic_Info_Changed', na=False)
].copy()

# Identify File_Closed events
close_events = usnjrnl[
    usnjrnl['EventInfo'].str.contains('File_Closed', na=False)
].copy()

print(f"Basic_Info_Changed events: {len(bdp_candidates):,}")
print(f"File_Closed events: {len(close_events):,}")

if len(close_events) > 0:
    # Merge on FileReferenceNumber (creates all pairs for same file)
    print("Merging events by FileReferenceNumber...")
    bdp_check = bdp_candidates.reset_index().merge(
        close_events[['FileReferenceNumber', 'EventTime']],
        on='FileReferenceNumber',
        how='inner',
        suffixes=('_bdp', '_close')
    )
    
    print(f"Checking {len(bdp_check):,} pairs...")
    
    # Calculate time difference (vectorized)
    bdp_check['time_diff'] = abs(
        (bdp_check['EventTime_bdp'] - bdp_check['EventTime_close']).dt.total_seconds()
    )
    
    # Find Basic_Info_Changed events with File_Closed within 1 second
    bdp_indices = bdp_check[bdp_check['time_diff'] <= 1]['index'].unique()
    
    print(f"BDP pairs found: {len(bdp_indices):,}")
    
    # Mark in original dataframe
    usnjrnl['is_bdp'] = 0
    usnjrnl.loc[bdp_indices, 'is_bdp'] = 1
else:
    usnjrnl['is_bdp'] = 0
    print("No File_Closed events found")

bdp_count = usnjrnl['is_bdp'].sum()
print(f"BDP events detected: {bdp_count:,} / {len(bdp_candidates):,} Basic_Info_Changed events")



Step 1: BDP Detection (optimized)...


Basic_Info_Changed events: 5,045
File_Closed events: 7,638
Merging events by FileReferenceNumber...
Checking 1,050,096 pairs...
BDP pairs found: 4,824
BDP events detected: 4,824 / 5,045 Basic_Info_Changed events


In [6]:
# Cell 5.5: Check available columns

print("Available columns in usnjrnl:")
print(usnjrnl.columns.tolist())
print(f"\nDataFrame shape: {usnjrnl.shape}")


Available columns in usnjrnl:
['TimeStamp(UTC+8)', 'USN', 'File/Directory Name', 'FullPath', 'EventInfo', 'SourceInfo', 'FileAttribute', 'FileReferenceNumber', 'FileName_MFT', 'SI_CreationTime_Formatted', 'SI_ModifiedTime_Formatted', 'SI_AccessedTime_Formatted', 'SI_MFTModifiedTime_Formatted', 'EventTime_Formatted', 'MFT_RecordNumber', 'EventTime', 'SI_CreationTime', 'SI_ModifiedTime', 'SI_AccessedTime', 'is_bdp']

DataFrame shape: (16647, 20)


In [7]:
# Cell 6: Group by file and calculate basic features - OPTIMIZED

print("\nGrouping events by file (vectorized)...")

# Build aggregation dict based on available columns
print("Step 1/5: Basic aggregation...")

agg_dict = {'EventInfo': 'count'}

# Add columns that exist
for col in ['FullPath', 'File/Directory Name', 'FileName_MFT', 
            'SI_CreationTime', 'SI_ModifiedTime', 'SI_AccessedTime']:
    if col in usnjrnl.columns:
        agg_dict[col] = 'first'

file_features_df = usnjrnl.groupby('FileReferenceNumber').agg(agg_dict)

# Rename columns to standard names
rename_map = {'EventInfo': 'file_event_count'}
if 'FullPath' in file_features_df.columns:
    rename_map['FullPath'] = 'file_path'
if 'File/Directory Name' in file_features_df.columns:
    rename_map['File/Directory Name'] = 'file_name'
if 'FileName_MFT' in file_features_df.columns and 'file_name' not in rename_map.values():
    rename_map['FileName_MFT'] = 'file_name'

file_features_df = file_features_df.rename(columns=rename_map)

print(f"  {len(file_features_df):,} files")

# Creation events
print("Step 2/5: Creation event detection...")
creation_mask = usnjrnl['EventInfo'].str.contains('File_Created', na=False)
if creation_mask.any():
    creation_by_file = usnjrnl[creation_mask].groupby('FileReferenceNumber').agg({
        'EventTime': 'first'
    }).rename(columns={'EventTime': 'file_creation_time'})
    
    file_features_df = file_features_df.join(creation_by_file, how='left')
    file_features_df['file_has_creation'] = file_features_df['file_creation_time'].notna().astype(int)
else:
    file_features_df['file_creation_time'] = None
    file_features_df['file_has_creation'] = 0

print(f"  {file_features_df['file_has_creation'].sum():,} files with creation event")

# BDP events
print("Step 3/5: BDP event detection...")
if usnjrnl['is_bdp'].sum() > 0:
    bdp_by_file = usnjrnl[usnjrnl['is_bdp'] == 1].groupby('FileReferenceNumber').agg({
        'EventTime': 'max'
    }).rename(columns={'EventTime': 'file_last_bdp_timestamp'})
    
    file_features_df = file_features_df.join(bdp_by_file, how='left')
    file_features_df['file_has_bdp'] = file_features_df['file_last_bdp_timestamp'].notna().astype(int)
else:
    file_features_df['file_last_bdp_timestamp'] = None
    file_features_df['file_has_bdp'] = 0

print(f"  {file_features_df['file_has_bdp'].sum():,} files with BDP")

# Zero nanoseconds (milliseconds = 0)
print("Step 4/5: Zero nanoseconds detection...")
if 'SI_CreationTime' in file_features_df.columns:
    file_features_df['zero_nanoseconds'] = (
        file_features_df['SI_CreationTime'].dt.microsecond == 0
    ).fillna(False).astype(int)
else:
    file_features_df['zero_nanoseconds'] = 0

print(f"  {file_features_df['zero_nanoseconds'].sum():,} files with zero nanoseconds")

print("\nStep 5/5: Feature creation complete")
print(f"\nFile-level features created for {len(file_features_df):,} files")
print(f"\nColumns: {list(file_features_df.columns)}")



Grouping events by file (vectorized)...
Step 1/5: Basic aggregation...
  11,813 files
Step 2/5: Creation event detection...
  11,291 files with creation event
Step 3/5: BDP event detection...
  842 files with BDP
Step 4/5: Zero nanoseconds detection...
  24 files with zero nanoseconds

Step 5/5: Feature creation complete

File-level features created for 11,813 files

Columns: ['file_event_count', 'file_path', 'file_name', 'FileName_MFT', 'SI_CreationTime', 'SI_ModifiedTime', 'SI_AccessedTime', 'file_creation_time', 'file_has_creation', 'file_last_bdp_timestamp', 'file_has_bdp', 'zero_nanoseconds']


In [8]:
# Cell 7: Step 3 - Tunneling Detection (FIXED - was creating 90M pairs)

print("\nStep 3: Tunneling Detection (optimized to prevent 90M pairs)...")

# Get creation and delete/rename events
creation_events = usnjrnl[
    usnjrnl['EventInfo'].str.contains('File_Created', na=False)
].copy()

delete_rename_events = usnjrnl[
    usnjrnl['EventInfo'].str.contains('File_Deleted|File_Renamed', na=False)
].copy()

print(f"Creation events: {len(creation_events):,}")
print(f"Delete/Rename events: {len(delete_rename_events):,}")

# OPTIMIZATION: Keep only FIRST creation per path (not per FRN)
# Tunneling detection only needs to check one creation event per path
creation_events_sorted = creation_events.sort_values('EventTime')
creation_first_per_path = creation_events_sorted.groupby('FullPath').first().reset_index()

print(f"First creation per path: {len(creation_first_per_path):,} (reduced from {len(creation_events):,})")

# OPTIMIZATION: Keep only LAST delete/rename per path
# We only care about the most recent delete before creation
delete_rename_sorted = delete_rename_events.sort_values('EventTime')
delete_rename_last_per_path = delete_rename_sorted.groupby('FullPath').last().reset_index()

print(f"Last delete/rename per path: {len(delete_rename_last_per_path):,} (reduced from {len(delete_rename_events):,})")

# Now merge - should create FAR fewer pairs
tunneling_candidates = creation_first_per_path.merge(
    delete_rename_last_per_path,
    on='FullPath',
    how='inner',
    suffixes=('_created', '_deleted')
)

print(f"Same-path pairs: {len(tunneling_candidates):,} (was 90M!)")

# Filter to DIFFERENT files (different FileReferenceNumbers)
tunneling_candidates = tunneling_candidates[
    tunneling_candidates['FileReferenceNumber_created'] != 
    tunneling_candidates['FileReferenceNumber_deleted']
]

print(f"Different-file pairs: {len(tunneling_candidates):,}")

if len(tunneling_candidates) > 0:
    # Calculate time difference (creation - deletion)
    tunneling_candidates['time_diff'] = (
        tunneling_candidates['EventTime_created'] - 
        tunneling_candidates['EventTime_deleted']
    ).dt.total_seconds()
    
    # Tunneling = delete/rename 0-15 seconds BEFORE creation
    tunneling_pairs = tunneling_candidates[
        (tunneling_candidates['time_diff'] > 0) &
        (tunneling_candidates['time_diff'] <= 15)
    ]
    
    print(f"Pairs within 15-second window: {len(tunneling_pairs):,}")
    
    if len(tunneling_pairs) > 0:
        # SANITY CHECK: SI-C diff must be < 1 day (86400 seconds)
        tunneling_pairs['si_c_diff'] = abs(
            (tunneling_pairs['SI_CreationTime_created'] - 
             tunneling_pairs['EventTime_created']).dt.total_seconds()
        )
        
        valid_tunneling = tunneling_pairs[
            tunneling_pairs['si_c_diff'] <= 86400
        ]
        
        print(f"Valid tunneling (SI-C diff < 1 day): {len(valid_tunneling):,}")
        
        tunneling_frns = set(valid_tunneling['FileReferenceNumber_created'].unique())
    else:
        tunneling_frns = set()
else:
    tunneling_frns = set()

# Add tunneling flag to file features
file_features_df['file_has_tunneling'] = file_features_df.index.isin(tunneling_frns).astype(int)

print(f"Files with tunneling: {file_features_df['file_has_tunneling'].sum():,}")



Step 3: Tunneling Detection (optimized to prevent 90M pairs)...
Creation events: 11,733
Delete/Rename events: 484
First creation per path: 5,754 (reduced from 11,733)
Last delete/rename per path: 206 (reduced from 484)
Same-path pairs: 177 (was 90M!)
Different-file pairs: 56
Pairs within 15-second window: 0
Files with tunneling: 0


In [9]:
# Cell 8: Step 4 - Behavior Counter Calculation (IMPROVED)

print("\nStep 4: Behavior Counter Calculation (with timestamp clustering)...")

# STEP 1: Detect timestamp clustering (multiple files sharing same SI_CreationTime)
print("  Detecting timestamp clusters...")
timestamp_counts = file_features_df['SI_CreationTime'].value_counts()
file_features_df['same_timestamp_count'] = file_features_df['SI_CreationTime'].map(timestamp_counts).fillna(1)

# Files in suspicious clusters (3+ files with identical timestamp)
suspicious_clusters = (file_features_df['same_timestamp_count'] >= 3).sum()
print(f"  Files in suspicious clusters (3+ files, same timestamp): {suspicious_clusters:,}")

# STEP 2: Calculate behavior counter
file_features_df['file_behavior_counter'] = 0

for frn in file_features_df.index:
    counter = 0
    file_info = file_features_df.loc[frn]
    
    si_creation_time = file_info['SI_CreationTime']
    
    if pd.isna(si_creation_time):
        file_features_df.loc[frn, 'file_behavior_counter'] = 0
        continue
    
    # Indicator 1: Creation event time discrepancy (INCREASED THRESHOLD: 60s)
    creation_discrepancy = False
    if file_info['file_has_creation'] == 1 and pd.notna(file_info['file_creation_time']):
        creation_diff = abs((si_creation_time - file_info['file_creation_time']).total_seconds())
        if creation_diff > 60:  # CHANGED: 5s → 60s
            creation_discrepancy = True
            counter += 1
    
    # Indicator 2: BDP timestamp discrepancy (INCREASED THRESHOLD: 60s)
    bdp_discrepancy = False
    if file_info['file_has_bdp'] == 1 and pd.notna(file_info['file_last_bdp_timestamp']):
        bdp_diff = abs((si_creation_time - file_info['file_last_bdp_timestamp']).total_seconds())
        if bdp_diff > 60:  # CHANGED: 5s → 60s
            bdp_discrepancy = True
            counter += 1
    
    # Indicator 3: Zero nanoseconds (STRONG SIGNAL)
    has_zero_ns = file_info['zero_nanoseconds'] == 1
    if has_zero_ns:
        counter += 1
    
    # Indicator 4: Timestamp clustering (SMOKING GUN - Oh et al.'s key pattern)
    in_cluster = file_info['same_timestamp_count'] >= 3
    if in_cluster and has_zero_ns:
        counter += 2  # Multiple files with SAME zero-ns timestamp = very suspicious
    
    # Indicator 5: Tunneling adjustment
    has_tunneling = file_info['file_has_tunneling'] == 1
    
    # IMPROVED LOGIC: Only penalize if MULTIPLE indicators + NO tunneling explanation
    if counter >= 2 and not has_tunneling:
        counter += 1  # Confirmed suspicious
    elif counter >= 1 and has_tunneling:
        counter -= 1  # Tunneling explains the discrepancy
    
    file_features_df.loc[frn, 'file_behavior_counter'] = counter

# Summary
print(f"\nBehavior counter distribution:")
print(file_features_df['file_behavior_counter'].value_counts().sort_index())

high_suspicion = (file_features_df['file_behavior_counter'] >= 3).sum()
print(f"\nFiles with counter >= 3: {high_suspicion:,} (EXPECTED: ~12)")



Step 4: Behavior Counter Calculation (with timestamp clustering)...
  Detecting timestamp clusters...
  Files in suspicious clusters (3+ files, same timestamp): 2,696

Behavior counter distribution:
file_behavior_counter
0    8359
1    3423
3      29
4       2
Name: count, dtype: int64

Files with counter >= 3: 31 (EXPECTED: ~12)


In [10]:
# Cell 9: Save UsnJrnl file features

print("\nSaving UsnJrnl file features...")
file_features_df.to_csv(USNJRNL_FEATURES_PATH)
print(f"Saved to: {USNJRNL_FEATURES_PATH}")
print(f"File size: {USNJRNL_FEATURES_PATH.stat().st_size / 1024:.2f} KB")
print(f"\nFeatures shape: {file_features_df.shape}")
print(f"Columns: {list(file_features_df.columns)}")



Saving UsnJrnl file features...
Saved to: /Users/soni/Github/Digital-Detectives_Thesis/notebooks/LW Testing Pipeline/Outputs/v2 filtered/LoneWolf_UsnJrnl_file_features.csv
File size: 2659.38 KB

Features shape: (11813, 15)
Columns: ['file_event_count', 'file_path', 'file_name', 'FileName_MFT', 'SI_CreationTime', 'SI_ModifiedTime', 'SI_AccessedTime', 'file_creation_time', 'file_has_creation', 'file_last_bdp_timestamp', 'file_has_bdp', 'zero_nanoseconds', 'file_has_tunneling', 'same_timestamp_count', 'file_behavior_counter']


In [11]:
# Cell 10: Load filtered LogFile

print("\n" + "="*60)
print("LOGFILE FILE-LEVEL FEATURES")
print("="*60)

print("\nLoading filtered LogFile...")
logfile = pd.read_csv(LOGFILE_FILTERED_PATH, low_memory=False)

print(f"Total events: {len(logfile):,}")
print(f"Unique files (Full Path): {logfile['Full Path'].nunique():,}")
print(f"\nEvent types:")
print(logfile['Event'].value_counts())



LOGFILE FILE-LEVEL FEATURES

Loading filtered LogFile...
Total events: 9,898
Unique files (Full Path): 2,149

Event types:
Event
Updating Modified Time                             4474
File Deletion                                      2259
File Creation                                      2126
Time Reversal Event                                 573
File Creation(File System Tunneling)                439
Time Reversal Event & Changing FileAttribute         14
Updating Modified Time & Changing FileAttribute      13
Name: count, dtype: int64


# Part 2: LogFile File-Level Features

In [12]:
# Cell 11: Parse LogFile Detail column (before -> after timestamps)

print("\nParsing LogFile Detail column...")

def parse_logfile_detail(detail_str):
    """
    Parse LogFile Detail column to extract before/after timestamps
    Example: "CreationTime : 2023-01-05 22:09:14 -> 2019-12-07 17:03:44(Zero in 100-nanoseconds)"
    Returns: dict with 'timestamp_type', 'before', 'after', 'zero_nanoseconds'
    """
    if pd.isna(detail_str):
        return None
    
    result = {
        'timestamp_type': None,
        'before': None,
        'after': None,
        'zero_nanoseconds': 0
    }
    
    # Check for zero nanoseconds
    if 'Zero in 100-nanoseconds' in detail_str or 'zero in 100-nanoseconds' in detail_str.lower():
        result['zero_nanoseconds'] = 1
    
    # Extract timestamp type
    if 'CreationTime' in detail_str:
        result['timestamp_type'] = 'CreationTime'
    elif 'ModifiedTime' in detail_str:
        result['timestamp_type'] = 'ModifiedTime'
    
    # Extract before -> after
    if '->' in detail_str:
        parts = detail_str.split('->')
        if len(parts) == 2:
            # Extract timestamp from "CreationTime : 2023-01-05 22:09:14"
            before_part = parts[0].strip()
            if ':' in before_part:
                before_ts = ':'.join(before_part.split(':')[1:]).strip()
                result['before'] = before_ts
            
            # Extract timestamp from "2019-12-07 17:03:44(Zero...)"
            after_part = parts[1].strip().split('(')[0].strip()
            result['after'] = after_part
    
    return result

# Apply parsing
logfile['parsed_detail'] = logfile['Detail'].apply(parse_logfile_detail)

print(f"Parsed details: {logfile['parsed_detail'].notna().sum():,} / {len(logfile):,}")



Parsing LogFile Detail column...
Parsed details: 5,450 / 9,898


In [13]:
# Cell 12: Group by file path and calculate LogFile features

print("\nGrouping LogFile events by file path...")

logfile_features = []

for file_path, group in logfile.groupby('Full Path'):
    
    # Basic file info
    file_name = group['File/Directory Name'].iloc[0] if 'File/Directory Name' in group.columns else None
    
    # Event counts
    event_count = len(group)
    
    # Parse details
    parsed_details = [d for d in group['parsed_detail'] if d is not None]
    
    # Count timestamp changes
    si_c_changes = sum(1 for d in parsed_details if d['timestamp_type'] == 'CreationTime')
    si_m_changes = sum(1 for d in parsed_details if d['timestamp_type'] == 'ModifiedTime')
    
    # Zero nanoseconds
    zero_nanoseconds = max([d['zero_nanoseconds'] for d in parsed_details]) if parsed_details else 0
    
    # Timestamp reversal direction
    reversal_to_past = group['Event'].str.contains('Time Reversal Event', na=False).any()
    
    # Simple behavior counter for LogFile
    # If Time Reversal Event detected, counter = 3
    behavior_counter = 3 if reversal_to_past else 0
    
    logfile_features.append({
        'file_path': file_path,
        'file_name': file_name,
        'logfile_event_count': event_count,
        'logfile_si_c_change_count': si_c_changes,
        'logfile_si_m_change_count': si_m_changes,
        'logfile_time_reversal': 1 if reversal_to_past else 0,
        'logfile_zero_nanoseconds': zero_nanoseconds,
        'logfile_behavior_counter': behavior_counter
    })

logfile_features_df = pd.DataFrame(logfile_features)

print(f"LogFile file-level features created for {len(logfile_features_df):,} files")
print(f"\nFeature summary:")
print(f"  Files with Time Reversal Event: {logfile_features_df['logfile_time_reversal'].sum():,}")
print(f"  Files with zero nanoseconds: {logfile_features_df['logfile_zero_nanoseconds'].sum():,}")
print(f"  Files with SI-C changes: {(logfile_features_df['logfile_si_c_change_count'] > 0).sum():,}")



Grouping LogFile events by file path...
LogFile file-level features created for 2,149 files

Feature summary:
  Files with Time Reversal Event: 272
  Files with zero nanoseconds: 45
  Files with SI-C changes: 234


In [14]:
# Cell 13: Save LogFile file features

print("\nSaving LogFile file features...")
logfile_features_df.to_csv(LOGFILE_FEATURES_PATH, index=False)
print(f"Saved to: {LOGFILE_FEATURES_PATH}")
print(f"File size: {LOGFILE_FEATURES_PATH.stat().st_size / 1024:.2f} KB")
print(f"\nFeatures shape: {logfile_features_df.shape}")
print(f"Columns: {list(logfile_features_df.columns)}")



Saving LogFile file features...
Saved to: /Users/soni/Github/Digital-Detectives_Thesis/notebooks/LW Testing Pipeline/Outputs/v2 filtered/LoneWolf_LogFile_file_features.csv
File size: 278.97 KB

Features shape: (2149, 8)
Columns: ['file_path', 'file_name', 'logfile_event_count', 'logfile_si_c_change_count', 'logfile_si_m_change_count', 'logfile_time_reversal', 'logfile_zero_nanoseconds', 'logfile_behavior_counter']


# Part 3: Merge UsnJrnl + LogFile Features

In [15]:
# Cell 14: Merge on file path

print("\n" + "="*60)
print("MERGING USNJRNL + LOGFILE FEATURES")
print("="*60)

print("\nMerging features on file path...")

# Reset index for UsnJrnl features to use file_path for merge
usnjrnl_for_merge = file_features_df.reset_index()

# Merge on file_path (OUTER JOIN - keep files from either source)
combined = usnjrnl_for_merge.merge(
    logfile_features_df,
    on='file_path',
    how='outer',
    suffixes=('_usnjrnl', '_logfile')
)

print(f"UsnJrnl files: {len(usnjrnl_for_merge):,}")
print(f"LogFile files: {len(logfile_features_df):,}")
print(f"Combined files: {len(combined):,}")

# Fill NaN values
combined['has_usnjrnl_events'] = combined['file_event_count'].notna().astype(int)
combined['has_logfile_events'] = combined['logfile_event_count'].notna().astype(int)
combined['has_both_sources'] = ((combined['has_usnjrnl_events'] == 1) & 
                                  (combined['has_logfile_events'] == 1)).astype(int)

print(f"\nSource distribution:")

# Calculate counts separately to avoid f-string syntax issues
usnjrnl_only = ((combined['has_usnjrnl_events'] == 1) & (combined['has_logfile_events'] == 0)).sum()
logfile_only = ((combined['has_usnjrnl_events'] == 0) & (combined['has_logfile_events'] == 1)).sum()
both_sources = combined['has_both_sources'].sum()

print(f"  UsnJrnl only: {usnjrnl_only:,}")
print(f"  LogFile only: {logfile_only:,}")
print(f"  Both sources: {both_sources:,}")



MERGING USNJRNL + LOGFILE FEATURES

Merging features on file path...
UsnJrnl files: 11,813
LogFile files: 2,149
Combined files: 13,569

Source distribution:
  UsnJrnl only: 11,324
  LogFile only: 1,756
  Both sources: 489


In [16]:
# Cell 15: Calculate combined features

print("\nCalculating combined features...")

# Max behavior counter from both sources
combined['file_behavior_counter'] = combined['file_behavior_counter'].fillna(0)
combined['logfile_behavior_counter'] = combined['logfile_behavior_counter'].fillna(0)
combined['max_behavior_counter'] = combined[['file_behavior_counter', 'logfile_behavior_counter']].max(axis=1)

# Combined zero nanoseconds
combined['zero_nanoseconds'] = combined['zero_nanoseconds'].fillna(0)
combined['logfile_zero_nanoseconds'] = combined['logfile_zero_nanoseconds'].fillna(0)
combined['zero_nanoseconds_combined'] = ((combined['zero_nanoseconds'] == 1) | 
                                          (combined['logfile_zero_nanoseconds'] == 1)).astype(int)

# Combined suspicion score (weighted average)
combined['combined_suspicion_score'] = (
    combined['file_behavior_counter'] * 0.6 + 
    combined['logfile_behavior_counter'] * 0.4
)

print(f"\nMax behavior counter distribution:")
print(combined['max_behavior_counter'].value_counts().sort_index())

print(f"\nFiles with zero nanoseconds (combined): {combined['zero_nanoseconds_combined'].sum():,}")



Calculating combined features...

Max behavior counter distribution:
max_behavior_counter
0.0    9846
1.0    3422
3.0     299
4.0       2
Name: count, dtype: int64

Files with zero nanoseconds (combined): 67


In [17]:
# Cell 16: Select final feature columns

print("\nSelecting final feature columns...")

# Select columns for final output
final_columns = [
    'FileReferenceNumber',
    'file_path',
    'file_name_usnjrnl',
    'file_name_logfile',
    
    # Source indicators
    'has_usnjrnl_events',
    'has_logfile_events',
    'has_both_sources',
    
    # UsnJrnl features
    'file_event_count',
    'file_has_creation',
    'file_has_bdp',
    'file_has_tunneling',
    'file_behavior_counter',
    'zero_nanoseconds',
    
    # LogFile features
    'logfile_event_count',
    'logfile_si_c_change_count',
    'logfile_si_m_change_count',
    'logfile_time_reversal',
    'logfile_zero_nanoseconds',
    'logfile_behavior_counter',
    
    # Combined features
    'max_behavior_counter',
    'zero_nanoseconds_combined',
    'combined_suspicion_score'
]

# Keep only columns that exist
final_columns_exist = [col for col in final_columns if col in combined.columns]
combined_final = combined[final_columns_exist].copy()

print(f"Final feature set: {combined_final.shape}")
print(f"Columns: {len(combined_final.columns)}")



Selecting final feature columns...
Final feature set: (13569, 22)
Columns: 22


In [18]:
# Cell 17: Save combined features

print("\nSaving combined features...")
combined_final.to_csv(COMBINED_FEATURES_PATH, index=False)
print(f"Saved to: {COMBINED_FEATURES_PATH}")
print(f"File size: {COMBINED_FEATURES_PATH.stat().st_size / 1024:.2f} KB")



Saving combined features...
Saved to: /Users/soni/Github/Digital-Detectives_Thesis/notebooks/LW Testing Pipeline/Outputs/v2 filtered/LoneWolf_combined_features.csv
File size: 2039.95 KB


In [19]:
# Cell 18: Final summary

print("\n" + "="*60)
print("PHASE 2 COMPLETE - FEATURE ENGINEERING SUMMARY")
print("="*60)

print(f"\nFiles processed: {len(combined_final):,}")

# Calculate counts separately to avoid f-string syntax issues
usnjrnl_only = ((combined_final['has_usnjrnl_events'] == 1) & (combined_final['has_logfile_events'] == 0)).sum()
logfile_only = ((combined_final['has_usnjrnl_events'] == 0) & (combined_final['has_logfile_events'] == 1)).sum()
both_sources = combined_final['has_both_sources'].sum()

print(f"  UsnJrnl only: {usnjrnl_only:,}")
print(f"  LogFile only: {logfile_only:,}")
print(f"  Both sources: {both_sources:,}")

print(f"\nSuspicious indicators:")

# Calculate indicator counts separately
counter_2_plus = (combined_final['max_behavior_counter'] >= 2).sum()
counter_3_plus = (combined_final['max_behavior_counter'] >= 3).sum()
zero_ns = combined_final['zero_nanoseconds_combined'].sum()
time_reversal = combined_final['logfile_time_reversal'].sum() if 'logfile_time_reversal' in combined_final.columns else 0

print(f"  Max behavior counter >= 2: {counter_2_plus:,}")
print(f"  Max behavior counter >= 3: {counter_3_plus:,}")
print(f"  Zero nanoseconds: {zero_ns:,}")
print(f"  Time Reversal Event: {time_reversal:,}")

print(f"\nOutput files:")
print(f"  UsnJrnl features: {USNJRNL_FEATURES_PATH}")
print(f"  LogFile features: {LOGFILE_FEATURES_PATH}")
print(f"  Combined features: {COMBINED_FEATURES_PATH}")

print("\nNext step: Analyze results and validate against ground truth (suspicious.csv)")



PHASE 2 COMPLETE - FEATURE ENGINEERING SUMMARY

Files processed: 13,569
  UsnJrnl only: 11,324
  LogFile only: 1,756
  Both sources: 489

Suspicious indicators:
  Max behavior counter >= 2: 301
  Max behavior counter >= 3: 301
  Zero nanoseconds: 67
  Time Reversal Event: 272.0

Output files:
  UsnJrnl features: /Users/soni/Github/Digital-Detectives_Thesis/notebooks/LW Testing Pipeline/Outputs/v2 filtered/LoneWolf_UsnJrnl_file_features.csv
  LogFile features: /Users/soni/Github/Digital-Detectives_Thesis/notebooks/LW Testing Pipeline/Outputs/v2 filtered/LoneWolf_LogFile_file_features.csv
  Combined features: /Users/soni/Github/Digital-Detectives_Thesis/notebooks/LW Testing Pipeline/Outputs/v2 filtered/LoneWolf_combined_features.csv

Next step: Analyze results and validate against ground truth (suspicious.csv)
